### Margin -Based Models - Linear SVM and SGD


### 1. Setup

import paths

In [2]:
from __future__ import annotations

import hashlib, json, re, sys, time, warnings
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (accuracy_score, average_precision_score,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GroupKFold

from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier

warnings.filterwarnings("ignore")

QUICK = False
FORCE_RETRAIN = False
SEED = 3244

ROOT = Path.cwd()
while not (ROOT / "data" / "splits").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data" / "splits").exists(), "run pipeline/notebooks/01-04 first"
sys.path.insert(0, str(ROOT / "pipeline"))
from load import load_split, load_lexicon, MODEL_SAFE_FEATURES

#create new directories to store results, figures and trained model
RESULTS = ROOT / "results" / "margin-based"
FIGURES = ROOT / "figures" / "margin-based"
MODELS  = ROOT / "models" / "margin-based" 
for d in (RESULTS, FIGURES, MODELS):
    d.mkdir(parents=True, exist_ok=True)

TRAIN_ROWS, VAL_ROWS = (20_000, 8_000) if QUICK else (200_000, 60_000)
N_UNIGRAMS, N_BIGRAMS, MIN_TERM_COUNT = 220, 80, 200
TARGET_PRECISION = 0.75
TRANSFER_TARGETS = ["figlang_reddit", "figlang_twitter",
                    "tweeteval_irony", "news_headlines"]

print(f"root      {ROOT}")
print(f"mode      {'QUICK' if QUICK else 'FULL'}  train={TRAIN_ROWS:,} val={VAL_ROWS:,}")

root      c:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1
mode      FULL  train=200,000 val=60,000


In [3]:
#ensure reprodcucibility using same version of libraries

import importlib.metadata as md_
VERSIONS = {p: md_.version(p) for p in
            ["scikit-learn", "pandas", "numpy", "joblib"]}
VERSIONS["python"] = sys.version.split()[0]
pd.Series(VERSIONS).to_frame("version")

,version
scikit-learn,1.9.0
pandas,3.0.6
numpy,2.4.6
joblib,1.5.3
python,3.11.3


## Load Data

using sarc-random

subsampling data to keep models fast -> only use full train data to train in final evaluation

only 20000 rows train data and 8000 validation used for this

In [19]:
train, val, test = load_split("sarc_random")
rng = np.random.default_rng(SEED)

if TRAIN_ROWS < len(train):
    train = train.iloc[rng.choice(len(train), TRAIN_ROWS, replace=False)].reset_index(drop=True)
if VAL_ROWS < len(val):
    val = val.iloc[rng.choice(len(val), VAL_ROWS, replace=False)].reset_index(drop=True)

print(f"train {len(train):,}   val {len(val):,}   test {len(test):,}") 
print(f"sarcastic rate — train {train.label.mean():.4f}  test {test.label.mean():.4f}") #check balance of sarcastic and non-saracstic data

pd.DataFrame(train)

train 200,000   val 60,000   test 137,065
sarcastic rate — train 0.5080  test 0.5107


,label,comment_clean,parent_clean,subreddit,author,score,ups,downs,ups_is_missing,downs_is_missing,...,parent_n_chars,parent_is_empty,parent_jaccard,len_ratio_to_parent,had_slash_s,had_label_hashtag,had_explicit_sarcasm_word,en_score,lang_method,parent_key
0,1,"Yep, even the women.",Big dick?,AskMen,arreayxn,6.0,6.0,0.0,False,False,...,9,False,0.000000,2.000000,False,False,False,1.000000,heuristic-high,956a10bb8f187f53
1,1,DEAL ME IN!,It really will be about tone. The fact that th...,politics,exalted0,24.0,24.0,0.0,False,False,...,152,False,0.038462,0.103448,False,False,False,1.000000,heuristic-high,8115563727a3c574
2,1,"Yeah, but your mum doesn't count dude.","""The screenplay features an original, heartfel...",Screenwriting,poseymund,4.0,4.0,0.0,False,False,...,55,False,0.000000,1.000000,False,False,False,0.900000,heuristic-high,d0ae3410602114a9
3,1,A Tight end obviously,Who do we draft to fix the Offensive line? Wha...,detroitlions,Perry87,15.0,15.0,0.0,False,False,...,227,False,0.024390,0.088889,False,False,False,0.973750,heuristic-high,9e3ea341dd93ff00
4,0,Like half a month actually.,they had even lost to the same astralis in kat...,GlobalOffensive,szogun_poland,4.0,4.0,0.0,False,False,...,87,False,0.050000,0.294118,False,False,False,1.000000,heuristic-high,ae9156f6796bb9b0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,1,Get police bots to stop them from stealing fro...,This is why there won't be any real economical...,Showerthoughts,GazLord,1.0,1.0,0.0,False,False,...,292,False,0.064516,0.368421,False,False,False,1.000000,heuristic-high,66d1baa2a7e01076
199996,0,VERY less racism,Slightly less racism?,funny,t3hlazy1,32.0,32.0,0.0,False,False,...,21,False,0.500000,1.000000,False,False,False,1.000000,heuristic-high,c71baafc9732d2d9
199997,0,Ted is the type of guy who let his wife have s...,TED CRUZ IS THE TYPE OF GUY TO PROMISE TO ENDO...,The_Donald,AmbitiousSimon,5.0,5.0,0.0,False,False,...,76,False,0.222222,1.333333,False,False,False,1.000000,heuristic-high,f4419edb2204198a
199998,1,kinda like gta v beta for last gen consoles,Advanced Warfare is a BETA isn't it? ;),CodAW,KingsBlade,3.0,3.0,0.0,False,False,...,39,False,0.066667,1.285714,False,False,False,0.844444,heuristic-high,89f458787d7a291a


### 2. Feature Engineering

#### Lexical Terms - Frequency of words in comment


SVM and Linear Regression can handle sparse data well compared to glassbox models - use TF-IDF to handle


TF-IDF measures how often a word appears in the comment compared to other comments  
TF = no. of times a certain word appears in a comment/total no. of words in the comment  
IDF = log(total number of comments/ number of comments containing the word)  
TF-IDF value = TF*IDF

It creates a new column for every word -> sparse column



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True) # up to phrases of 2 words
# fit and transform only on train 
train = vec.fit_transform(train["comment_clean"]) 
val = vec.transform(val["comment_clean"]) 
#only transform on validation (learn data from train and apply on validation) to prevent leakage

#scaling (TF-IDF columns are v small - L2 normalized)




print(train.shape) 

#sample of how data looks like after transformed
names = vec.get_feature_names_out()
pd.DataFrame(train[:5, :20].toarray(), columns=names[:20])


(200000, 114480)


,00,00 for,000,000 00,000 000,000 and,000 calories,000 feet,000 for,000 in,000 miles,000 of,000 on,000 people,000 per,000 refugees,000 so,000 times,000 year,000 years
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Lexicon Block

NRC features (columns 0-9): For each emotion, count how many unique words in the comment appear in that emotion's word set, divided by the comment's token count. Dividing by length (n) turns counts into rates, so long comments aren't favored. 

VADER features: Compute the reply's mean, positive share, and negative share, and the parent comment's mean sentiment (pm).


Incongruity features (the sarcasm-specific part):  
incongruity_pos_reply_neg_parent is 1 if the reply is positive (m > 0.05) while the parent is negative (pm < -0.05), else 0.  
 A cheerful reply to a negative comment is a classic sarcasm pattern ("Oh great, congrats on losing your job").

incongruity_signed is m * pm. It's negative when the signs differ (a strong mismatch), positive when they agree, and the magnitude captures how strong the sentiment is on both sides.


Linear models can't build interactions. A glass-box tree or EBM interaction term can discover "positive reply AND negative parent" on its own, but a linear model only adds up independent effect. -> incongruity features r important here




In [24]:

#same as glassbox models
EMOTIONS = ["anger", "anticipation", "disgust", "fear", "joy",
            "sadness", "surprise", "trust", "positive", "negative"]
LEXICON_NAMES = ([f"nrc_{e}" for e in EMOTIONS] +
                 ["vader_mean", "vader_pos_share", "vader_neg_share",
                  "parent_vader_mean", "incongruity_pos_reply_neg_parent",
                  "incongruity_signed"])

VADER = load_lexicon("vader")
_nrc  = load_lexicon("nrc")
NRC   = {e: _nrc.get(e, set()) for e in EMOTIONS}

def _vader(tk):
    vals = [VADER[w] for w in tk if w in VADER]
    if not vals:
        return 0.0, 0.0, 0.0
    a = np.asarray(vals, dtype=np.float32)
    return float(a.mean()), float((a > 0).mean()), float((a < 0).mean())

def lexicon_block(text, parent):
    out = np.zeros((len(text), len(LEXICON_NAMES)), dtype=np.float32)
    k = len(EMOTIONS)
    for r, (t, p) in enumerate(zip(text.fillna(""), parent.fillna(""))):
        tk = tokens(t); s = set(tk); n = max(1, len(tk))
        for j, e in enumerate(EMOTIONS):
            out[r, j] = len(s & NRC[e]) / n
        m, pos, neg = _vader(tk)
        pm, _, _ = _vader(tokens(p))
        out[r, k:k+6] = [m, pos, neg, pm,
                         1.0 if (m > 0.05 and pm < -0.05) else 0.0, m * pm]
    return out

print(f"{len(LEXICON_NAMES)} lexicon features | VADER {len(VADER):,} tokens")

16 lexicon features | VADER 7,506 tokens


In [ ]:
# fit them on train data

L_train = lexicon_block(train["comment_clean"], train["parent_clean"])
L_val = lexicon_block(val["comment_clean"], val["parent_clean"])


#toolazy to fix this error will be back


IndexError: Index dimension must be 1 or 2. Got 0

### Parameter Sweeping

Compare on validation:  
TF-IDF only  
TF-IDF + lexicon block  
Lexicon block only

hyperparameter tuning 
Linear SVM -> regularization , pick best c  
SGD -> pick the best learning rate, alpha